In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tools import get_metadata
from stats_tools import iqr_interval, pval_stars, readable_pval
from anais_jobs import concat_and_save
from multi_projects_jobs import detect_ecg_job
from icca_jobs import resample_pse_tt_job, resample_clinical_job, icca_csf_job
from icca_tools import get_csf_subs

In [ ]:
data = concat_and_save(save = False)

In [ ]:
data.head()

In [ ]:
# --- ICP distribution: screening for aberrant values ---
icp = data['icp'].dropna()

ink, ink2 = '#1f2933', '#6b7280'
bleu, rouge = '#3b6ea5', '#c0392b'

fig, axs = plt.subplots(ncols=3, figsize=(15, 4.2), constrained_layout=True)

# 1. full range, log y: otherwise the tail is invisible
ax = axs[0]
ax.hist(icp, bins=np.arange(np.floor(icp.min()), icp.max() + 2, 2), color=bleu, edgecolor='none')
ax.set_yscale('log')
ax.axvspan(icp.min() - 1, 0, color=rouge, alpha=.10)
ax.axvspan(60, icp.max() + 1, color=rouge, alpha=.10)
ax.set_title('ICP, full range (log y)', color=ink, fontsize=10)
ax.set_xlabel('ICP (mmHg)'); ax.set_ylabel('number of 60-min windows')

# 2. zoom on the physiological range
ax = axs[1]
zoom = icp[(icp >= -5) & (icp <= 60)]
ax.hist(zoom, bins=np.arange(0, 61, 1), color=bleu, edgecolor='none')
ax.axvline(icp.median(), color=ink, lw=2)
ax.axvline(20, color=rouge, lw=2, ls='--')
ax.text(icp.median(), ax.get_ylim()[1] * .95, f'  median {icp.median():.0f}', color=ink, fontsize=9, va='top')
ax.text(20, ax.get_ylim()[1] * .80, '  threshold 20', color=rouge, fontsize=9, va='top')
ax.set_title('zoom -5 to 60 mmHg', color=ink, fontsize=10)
ax.set_xlabel('ICP (mmHg)')

# 3. ICP vs pulse amplitude: a true ICP pulsates, an artefact does not
ax = axs[2]
ok = (data['icp'] >= -5) & (data['icp'] <= 60)
ax.scatter(data.loc[ok, 'icp'], data.loc[ok, 'icp_pulse_amplitude'], s=4, alpha=.15, color=bleu, edgecolors='none')
ax.scatter(data.loc[~ok, 'icp'], data.loc[~ok, 'icp_pulse_amplitude'], s=10, alpha=.7, color=rouge, edgecolors='none')
ax.axhline(0.5, color=ink2, lw=1, ls=':')
ax.text(ax.get_xlim()[1], 0.5, 'flat signal  ', color=ink2, fontsize=9, ha='right', va='bottom')
ax.set_title('ICP vs pulse amplitude', color=ink, fontsize=10)
ax.set_xlabel('ICP (mmHg)'); ax.set_ylabel('pulse amplitude (mmHg)')

for ax in axs:
    ax.grid(axis='y', color='#e5e7eb', lw=.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.tick_params(colors=ink2, labelsize=9)

fig.suptitle(f"ICP distribution — {len(icp)} windows, {data['subject'].nunique()} subjects",
             color=ink, fontsize=12)
plt.show()

# --- counts ---
for lab, m in [('ICP < -5', icp < -5), ('ICP > 60', icp > 60), ('ICP > 100', icp > 100)]:
    print(f'{lab:10} : {m.sum():5d} windows ({m.mean()*100:5.2f} %)')
print(f"\npercentiles : 1%={icp.quantile(.01):.1f}  50%={icp.median():.1f}  "
      f"99%={icp.quantile(.99):.1f}  99.9%={icp.quantile(.999):.1f}  max={icp.max():.1f}")
print(f"negative CPP : {(data['ppc'] < 0).sum()} windows, "
      f"of which {((data['ppc'] < 0) & (data['icp'] > 100)).sum()} with ICP > 100")

In [ ]:
# keep only physiological ICP values and a meaningful pulse amplitude
data = data[(data['icp'].between(-5, 60)) & (data['icp_pulse_amplitude'] > 0.5)]

In [ ]:
# --- respiratory rate distribution: screening for aberrant values ---
# same approach as for ICP. A rate of 2 or 47 cpm is not a patient, it is a failed cycle
# detection; these windows pollute everything that involves the phase.
fr = data['resp_rate'].dropna()

ink, ink2 = '#1f2933', '#6b7280'
bleu, rouge = '#3b6ea5', '#c0392b'
FR_MIN, FR_MAX = 5, 40          # candidate bounds, to adjust from the figure

fig, axs = plt.subplots(ncols=3, figsize=(15, 4.2), constrained_layout=True)

# 1. full range, log y: otherwise the tails are invisible
ax = axs[0]
ax.hist(fr, bins=np.arange(0, np.ceil(fr.max()) + 1, 1), color=bleu, edgecolor='none')
ax.set_yscale('log')
ax.axvspan(0, FR_MIN, color=rouge, alpha=.10)
ax.axvspan(FR_MAX, fr.max() + 1, color=rouge, alpha=.10)
ax.set_title('RR, full range (log y)', color=ink, fontsize=10)
ax.set_xlabel('respiratory rate (cpm)'); ax.set_ylabel('number of 60-min windows')

# 2. zoom on the plausible range, split by ventilation mode: under controlled ventilation the
# rate is set, hence narrow; under assisted ventilation it is more spread but stays physiological
ax = axs[1]
for mode, couleur in [('controlled', '#4b5563'), ('assisted', '#9ca3af')]:
    v = data.loc[data['ventilation_mode'] == mode, 'resp_rate'].dropna()
    ax.hist(v, bins=np.arange(0, 51, 1), color=couleur, alpha=.75, label=f'{mode} (n={len(v)})')
ax.axvline(FR_MIN, color=rouge, lw=2, ls='--'); ax.axvline(FR_MAX, color=rouge, lw=2, ls='--')
ax.axvline(fr.median(), color=ink, lw=2)
ax.text(fr.median(), ax.get_ylim()[1] * .95, f'  median {fr.median():.0f}',
        color=ink, fontsize=9, va='top')
ax.set_title('zoom 0 to 50 cpm by ventilation mode', color=ink, fontsize=10)
ax.set_xlabel('respiratory rate (cpm)'); ax.set_xlim(0, 50)
ax.legend(frameon=False, fontsize=8)

# 3. RR vs variability: a true rate is stable over one hour, a runaway detection gives a
# huge MAD. This is the counterpart of the ICP / pulse amplitude pair.
ax = axs[2]
ax.scatter(fr, data.loc[fr.index, 'resp_variability'], s=4, alpha=.15, color=bleu, edgecolors='none')
hors = (fr < FR_MIN) | (fr > FR_MAX)
ax.scatter(fr[hors], data.loc[fr[hors].index, 'resp_variability'], s=10, alpha=.7,
           color=rouge, edgecolors='none')
ax.axvline(FR_MIN, color=rouge, lw=1, ls='--'); ax.axvline(FR_MAX, color=rouge, lw=1, ls='--')
ax.set_title('RR vs within-window variability', color=ink, fontsize=10)
ax.set_xlabel('respiratory rate (cpm)'); ax.set_ylabel('variability (MAD, cpm)')

for ax in axs:
    ax.grid(axis='y', color='#e5e7eb', lw=.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.tick_params(colors=ink2, labelsize=9)

fig.suptitle(f"Respiratory rate distribution — {len(fr)} windows, "
             f"{data['subject'].nunique()} subjects", color=ink, fontsize=12)
plt.show()

# --- counts and choice of the bounds ---
print(f"percentiles : 0.1%={fr.quantile(.001):.1f}  1%={fr.quantile(.01):.1f}  "
      f"50%={fr.median():.1f}  99%={fr.quantile(.99):.1f}  99.9%={fr.quantile(.999):.1f}  "
      f"min={fr.min():.1f}  max={fr.max():.1f}")
print(f"variability (MAD) : median {data['resp_variability'].median():.2f} cpm, "
      f"99% = {data['resp_variability'].quantile(.99):.2f} cpm\n")

cout = []
for lo, hi in [(5, 40), (6, 35), (8, 30), (8, 35), (10, 30)]:
    garde = data['resp_rate'].between(lo, hi)
    cout.append({'bounds': f'{lo} - {hi} cpm',
                 'windows lost': int((~garde).sum()),
                 '% lost': round((~garde).mean() * 100, 2),
                 'subjects affected': int(data.loc[~garde, 'subject'].nunique()),
                 'subjects entirely lost': int(
                     (data.groupby('subject')['resp_rate'].apply(lambda s: s.between(lo, hi).sum()) == 0).sum())})
display(pd.DataFrame(cout).set_index('bounds'))
print("The next cell applies the filter: adjust FR_MIN / FR_MAX from this table.")

In [ ]:
# respiratory rate filter: bounds chosen from the previous cell
data = data[data['resp_rate'].between(FR_MIN, FR_MAX)]
print(f"{len(data)} windows remaining, {data['subject'].nunique()} subjects")

In [ ]:
# --- window enrichment: ICCA (sedation, Glasgow), heart rate and CSF drainage ---
# The design matrix contains neither the drug doses, nor the Glasgow, nor the heart rate, nor
# the drained CSF. They are recomputed window by window by rebuilding the time bounds: the
# 'time' column is the number of days since ICU admission, so start = entree_rea + time days.
ENRICHMENT_FILE = Path().cwd() / 'df_for_stats' / 'icca_enrichment.pkl'
RECOMPUTE = not ENRICHMENT_FILE.exists()        # set to True to force the recomputation
MEDICAMENTS = ['Propofol', 'Sufentanil', 'Midazolam']
DUREE_FENETRE_MIN = 60                          # must match delta_minutes of the design matrix

meta_tous = get_metadata().set_index('ID_pseudo')
debut_fen = (pd.to_datetime(data['subject'].map(meta_tous['entree_rea']))
             + pd.to_timedelta(data['time'], unit='D'))
fin_fen = debut_fen + pd.Timedelta(minutes=DUREE_FENETRE_MIN)

def agg_serie(da, d1, d2, fonction=np.nanmedian):
    """aggregate a dated DataArray (one-minute step) over each window [d1, d2["""
    dates = da[da.dims[0]].values.astype('datetime64[ns]')
    vals = np.asarray(da.values, dtype=float)
    i1, i2 = np.searchsorted(dates, d1), np.searchsorted(dates, d2)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')          # empty slices -> All-NaN slice
        return np.array([fonction(vals[a:b]) if b > a else np.nan for a, b in zip(i1, i2)])

if RECOMPUTE:
    csf_subs = set(get_csf_subs())
    lignes, echecs = [], {}
    for i_sub, (sub, g) in enumerate(data.groupby('subject')):
        d1 = debut_fen.loc[g.index].values.astype('datetime64[ns]')
        d2 = fin_fen.loc[g.index].values.astype('datetime64[ns]')
        res = pd.DataFrame(index=g.index)

        # syringe pump doses: a drug absent from the ICCA record is 0 (never given), an
        # unreadable ICCA record is NaN (missing information) -> the two are distinct
        try:
            ds_pse = resample_pse_tt_job.get(sub)
            for nom in MEDICAMENTS:
                res[nom.lower()] = (agg_serie(ds_pse[nom], d1, d2, np.nanmean)
                                    if nom in ds_pse else 0.0)
        except Exception as e:
            echecs.setdefault('pse', []).append(sub)
            for nom in MEDICAMENTS:
                res[nom.lower()] = np.nan

        # total Glasgow, rarely scored under deep sedation: many NaN expected
        try:
            ds_clin = resample_clinical_job.get(sub)
            res['glasgow_tot'] = (agg_serie(ds_clin['Glasgow, total'], d1, d2)
                                  if 'Glasgow, total' in ds_clin else np.nan)
        except Exception:
            echecs.setdefault('glasgow', []).append(sub)
            res['glasgow_tot'] = np.nan

        # heart rate: median of the RR intervals of the window
        try:
            pics = detect_ecg_job.get(sub).to_dataframe()
            dates_pics = pics['peak_date'].values.astype('datetime64[ns]')
            temps_pics = pics['peak_time'].values.astype(float)
            j1, j2 = np.searchsorted(dates_pics, d1), np.searchsorted(dates_pics, d2)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                res['heart_rate'] = [np.nanmedian(60 / np.diff(temps_pics[a:b])) if b - a > 2 else np.nan
                                     for a, b in zip(j1, j2)]
        except Exception:
            echecs.setdefault('ecg', []).append(sub)
            res['heart_rate'] = np.nan

        # CSF: volume drained during the window. NaN if the patient has no drain, 0 if a
        # drain is in place but nothing was drained over the hour: do not confuse the two
        if sub in csf_subs:
            try:
                ds_csf = icca_csf_job.get(sub)
                res['csf_ml'] = agg_serie(ds_csf['Vol vidé (E_S)'], d1, d2, np.nansum)
                res['csf_ml'] = res['csf_ml'].fillna(0.0)
            except Exception:
                echecs.setdefault('csf', []).append(sub)
                res['csf_ml'] = np.nan
        else:
            res['csf_ml'] = np.nan
        res['a_derivation'] = sub in csf_subs

        lignes.append(res)
        if (i_sub + 1) % 20 == 0:
            print(f'  {i_sub + 1} subjects processed')

    enrichi = pd.concat(lignes).sort_index()
    ENRICHMENT_FILE.parent.mkdir(exist_ok=True)
    enrichi.to_pickle(ENRICHMENT_FILE)
    for source, subs in echecs.items():
        print(f'{source} unavailable for {len(subs)} subjects : {subs[:8]}{" ..." if len(subs) > 8 else ""}')
else:
    enrichi = pd.read_pickle(ENRICHMENT_FILE)
    print(f'enrichment read back from {ENRICHMENT_FILE.name}')

# join on the window index: data keeps its columns, the new ones are added
data = data.join(enrichi[[c for c in enrichi.columns if c not in data.columns]])

cols_enrichies = ['propofol', 'sufentanil', 'midazolam', 'glasgow_tot', 'heart_rate', 'csf_ml']
dispo = pd.DataFrame({
    'windows available': [int(data[c].notna().sum()) for c in cols_enrichies],
    '% of windows': [round(data[c].notna().mean() * 100, 1) for c in cols_enrichies],
    'subjects available': [int(data.loc[data[c].notna(), 'subject'].nunique()) for c in cols_enrichies],
    'median when available': [round(float(data[c].median()), 2) for c in cols_enrichies],
}, index=cols_enrichies)
print(f"\n{len(data)} windows, {data['subject'].nunique()} subjects, "
      f"{data['a_derivation'].groupby(data['subject']).first().sum()} patients with CSF drainage")
display(dispo)

In [ ]:
# --- inspiratory vs expiratory peak windows: clinical and therapeutic description ---
from scipy.stats import mannwhitneyu, wilcoxon, chi2_contingency

def etoiles(p):
    """pval_stars returns 'ns' on a NaN, which would be mistaken for a true non-significant"""
    return pval_stars(p) if np.isfinite(p) else 'na'

def test_apparie(par_sujet):
    """paired Wilcoxon, NaN if too few pairs or if all differences are zero
    (scipy raises a ValueError in that case, typically an identical dose everywhere)"""
    if len(par_sujet) < 6:
        return np.nan
    try:
        return wilcoxon(par_sujet['inspiration'], par_sujet['expiration']).pvalue
    except ValueError:
        return np.nan

def chi2_robuste(tableau):
    """NaN rather than an exception if the table is degenerate (smaller than 2x2, zero margin)"""
    t = tableau.values if hasattr(tableau, 'values') else tableau
    if min(t.shape) < 2 or (t.sum(axis=0) == 0).any() or (t.sum(axis=1) == 0).any():
        return np.nan
    try:
        return chi2_contingency(t)[1]
    except ValueError:
        return np.nan

ordre_ie = ['inspiration', 'expiration']
palette_ie = {'inspiration': '#3b6ea5', 'expiration': '#c0392b'}
ie = data[data['label_max_icp_phase'].isin(ordre_ie)].copy()

variables_ie = [
    ('propofol',    'Propofol (mg/h)',            'A'),
    ('sufentanil',  'Sufentanil (ug/h)',          'B'),
    ('glasgow_tot', 'Glasgow total',              'C'),
    ('heart_rate',  'Heart rate (bpm)',           'D'),
    ('abp',         'ABP (mmHg)',                 'E'),
    ('ppc',         'CPP (mmHg)',                 'F'),
    ('csf_ml',      'CSF drained (mL/h)',         'G'),
    ('time',        'Days from ICU admission',    'H'),
]

# if the enrichment cell did not run, skip the missing variables rather than raising a
# KeyError in the middle of the figure
absentes = [c for c, _, _ in variables_ie if c not in data.columns]
if absentes:
    print(f'columns missing from data, panels skipped : {absentes}')
    variables_ie = [v for v in variables_ie if v[0] in data.columns]

rng_ie = np.random.default_rng(0)
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(15, 12.5), constrained_layout=True)
lignes_ie = []

for ax, (col, label, lettre) in zip(axs.flat, variables_ie):
    v = ie[[col, 'label_max_icp_phase', 'subject']].dropna()
    if v.empty:
        ax.text(.5, .5, f'{col} : no data', ha='center', va='center', transform=ax.transAxes)
        continue
    sns.violinplot(data=v, x='label_max_icp_phase', y=col, order=ordre_ie,
                   hue='label_max_icp_phase', palette=palette_ie, legend=False,
                   inner='quartile', cut=0, linewidth=1, ax=ax)
    for coll in ax.collections:
        coll.set_alpha(.30)
    for i, phase in enumerate(ordre_ie):
        pts = v.loc[v['label_max_icp_phase'] == phase, col].values
        if pts.size > 300:
            pts = rng_ie.choice(pts, 300, replace=False)
        ax.scatter(i + rng_ie.normal(0, .06, pts.size), pts, s=4, alpha=.25,
                   color=palette_ie[phase], edgecolor='none', zorder=3)

    x = v.loc[v['label_max_icp_phase'] == 'inspiration', col].values
    y = v.loc[v['label_max_icp_phase'] == 'expiration', col].values
    p = mannwhitneyu(x, y, alternative='two-sided').pvalue
    par_sujet = (v.pivot_table(index='subject', columns='label_max_icp_phase', values=col,
                               aggfunc='median').reindex(columns=ordre_ie).dropna())
    p_suj = test_apparie(par_sujet)

    lo, hi = ax.get_ylim(); h = hi + (hi - lo) * .03
    ax.plot([0, 0, 1, 1], [h, h + (hi - lo) * .02, h + (hi - lo) * .02, h], lw=1.2, color=ink)
    ax.text(.5, h + (hi - lo) * .035, f'{pval_stars(p)}  (p{readable_pval(p)})',
            ha='center', va='bottom', color=ink, fontsize=9)
    ax.set_ylim(lo, h + (hi - lo) * .14)
    ax.set_title(f'{lettre}. {label}', color=ink, fontsize=11, loc='left')
    ax.set_xlabel(''); ax.set_ylabel(label)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'{ph}\nn = {(v["label_max_icp_phase"] == ph).sum()}' for ph in ordre_ie])

    lignes_ie.append({'Variable': label,
                      'inspiration': f'{np.median(x):.2f} {iqr_interval(x, 2)}',
                      'expiration': f'{np.median(y):.2f} {iqr_interval(y, 2)}',
                      'n windows': len(v), 'n subjects': v['subject'].nunique(),
                      'p (windows)': p, 'stars': etoiles(p),
                      'p (paired, subject)': p_suj, 'stars paired': etoiles(p_suj)})

# I: sedation level, three-state ordinal variable -> distribution and chi2
ax = axs.flat[8]
ct_sed = pd.crosstab(ie['label_max_icp_phase'], ie['sedation']).reindex(ordre_ie).fillna(0)
pct_sed = ct_sed.div(ct_sed.sum(axis=1), axis=0) * 100
p_sed = chi2_robuste(ct_sed)
largeur = .8 / max(ct_sed.shape[1], 1)
gris = ['#4b5563', '#9ca3af', '#d1d5db', '#e5e7eb']
for j, niveau in enumerate(ct_sed.columns):
    ax.bar(np.arange(2) + (j - (ct_sed.shape[1] - 1) / 2) * largeur, pct_sed[niveau].values,
           width=largeur, color=gris[j % len(gris)], label=f'level {niveau:g}')
ax.set_ylim(0, 118)
ax.plot([0, 0, 1, 1], [104, 107, 107, 104], lw=1.2, color=ink)
ax.text(.5, 108, f'{etoiles(p_sed)} (p{readable_pval(p_sed)}, chi2)', ha='center',
        va='bottom', color=ink, fontsize=9)
ax.set_title('I. Sedation level', color=ink, fontsize=11, loc='left')
ax.set_ylabel('% of windows'); ax.set_xticks([0, 1]); ax.set_xticklabels(ordre_ie)
ax.legend(frameon=False, fontsize=8, loc='upper right')

fig.suptitle('Clinical and therapeutic context of inspiratory vs expiratory ICP peak',
             color=ink, fontsize=13)
plt.show()

if not ct_sed.empty:
    lignes_ie.append({'Variable': f'Sedation (% level {ct_sed.columns[-1]:g})',
                      'inspiration': f'{pct_sed.iloc[0, -1]:.0f} %',
                      'expiration': f'{pct_sed.iloc[1, -1]:.0f} %',
                      'n windows': int(ct_sed.values.sum()), 'n subjects': ie['subject'].nunique(),
                      'p (windows)': p_sed, 'stars': etoiles(p_sed),
                      'p (paired, subject)': np.nan, 'stars paired': 'na'})

table_ie = pd.DataFrame(lignes_ie).set_index('Variable')
for c in ['p (windows)', 'p (paired, subject)']:
    table_ie[c] = table_ie[c].map(lambda p: '' if not np.isfinite(p) else
                                  ('< 0.001' if p < 0.001 else f'{p:.3f}'))
print("Median [IQR]. Mann-Whitney on windows, chi2 for sedation, and subject-level paired "
      "Wilcoxon which corrects for the dependence of the windows of a given patient.")
display(table_ie)

In [ ]:
# --- patient level: who peaks in inspiration? skull, delay and admission diagnosis ---
# one patient = one observation, the only simple way not to count the same subject 200 times
from scipy.stats import kruskal, spearmanr

def kruskal_robuste(groupes):
    """NaN rather than an exception if fewer than two usable groups"""
    groupes = [g for g in groupes if len(g) > 0]
    if len(groupes) < 2 or all(len(np.unique(np.concatenate(groupes))) < 2 for _ in [0]):
        return np.nan
    try:
        return kruskal(*groupes).pvalue
    except ValueError:
        return np.nan

part_insp = (ie.groupby('subject')['label_max_icp_phase']
               .apply(lambda s: (s == 'inspiration').mean() * 100).rename('pct_inspiration'))
meta_ie = meta_tous.reindex(part_insp.index).join(part_insp)
meta_ie['skull_clean'] = meta_ie['Skull'].astype(str).str.strip().str.lower()
meta_ie.loc[meta_ie['skull_clean'].isin(['nan', 'none', '']), 'skull_clean'] = np.nan
meta_ie['motif_clean'] = meta_ie['motif'].astype(str).str.strip().str.upper()
meta_ie.loc[meta_ie['motif_clean'] == 'NAN', 'motif_clean'] = np.nan
meta_ie['delais_admission_j'] = meta_ie['delais']

fig, axs = plt.subplots(ncols=3, figsize=(16, 4.8), constrained_layout=True)

def compare_categorie(ax, colonne, titre, lettre, palette_cat):
    v = meta_ie.dropna(subset=[colonne, 'pct_inspiration'])
    ordre = [m for m in palette_cat if m in set(v[colonne])]
    if not ordre:
        ax.text(.5, .5, f'{colonne} : no data', ha='center', va='center',
                transform=ax.transAxes)
        return np.nan, v
    groupes = [v.loc[v[colonne] == m, 'pct_inspiration'].values for m in ordre]
    p = kruskal_robuste(groupes)
    sns.boxplot(data=v, x=colonne, y='pct_inspiration', order=ordre, hue=colonne,
                palette=palette_cat, legend=False, width=.55, showfliers=False, ax=ax)
    sns.stripplot(data=v, x=colonne, y='pct_inspiration', order=ordre, color=ink,
                  size=3.5, alpha=.6, jitter=.22, ax=ax)
    ax.axhline(50, color='#c0392b', lw=1, ls='--')
    etoile = pval_stars(p) if np.isfinite(p) else 'na'
    ax.set_title(f'{lettre}. {titre}\nKruskal-Wallis {etoile} (p{readable_pval(p)})',
                 color=ink, fontsize=11, loc='left')
    ax.set_xlabel(''); ax.set_ylabel('% of windows peaking in inspiration')
    ax.set_xticks(range(len(ordre)))
    ax.set_xticklabels([f'{m}\nn = {(v[colonne] == m).sum()}' for m in ordre], fontsize=9)
    return p, v

p_skull, v_skull = compare_categorie(
    axs[0], 'skull_clean', 'Skull status', 'A',
    {'close': '#9ca3af', 'craniotomy': '#3b6ea5', 'craniectomy': '#c0392b'})
p_motif, v_motif = compare_categorie(
    axs[1], 'motif_clean', 'Admission diagnosis', 'B',
    dict(zip(sorted(meta_ie['motif_clean'].dropna().unique()),
             sns.color_palette('Set2', meta_ie['motif_clean'].nunique()))))

ax = axs[2]
v_del = meta_ie.dropna(subset=['delais_admission_j', 'pct_inspiration'])
r_del = (spearmanr(v_del['delais_admission_j'], v_del['pct_inspiration'])
         if len(v_del) >= 3 else None)
ax.scatter(v_del['delais_admission_j'], v_del['pct_inspiration'], s=30, color='#3b6ea5',
           edgecolor='white')
ax.axhline(50, color='#c0392b', lw=1, ls='--')
titre_c = ('C. Delay from admission to monitoring\ninsufficient data' if r_del is None else
           f'C. Delay from admission to monitoring\nSpearman r = {r_del.statistic:.2f} '
           f'{pval_stars(r_del.pvalue)} (p{readable_pval(r_del.pvalue)}, n = {len(v_del)})')
ax.set_title(titre_c, color=ink, fontsize=11, loc='left')
ax.set_xlabel('Days from ICU admission to monitoring onset')
ax.set_ylabel('% of windows peaking in inspiration')

fig.suptitle('Who peaks in inspiration? One point per patient', color=ink, fontsize=13)
plt.show()

resume_meta = (meta_ie.groupby('skull_clean')['pct_inspiration']
                 .agg(patients='size', median='median',
                      q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75)).round(1))
display(resume_meta)
print(f"whole cohort : {part_insp.median():.0f} % {iqr_interval(part_insp, 0)} of windows "
      f"with an inspiratory peak per patient, over {len(part_insp)} patients")
print("The red line at 50 % separates inspiration-dominant patients from the others: "
      "a patient above it spends most of the monitoring with an ICP peak in inspiration.")

In [ ]:
# --- clustering of the windows: feature matrix ---
# goal: see whether cerebral compliance profiles emerge from the 60-min windows
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

ink = '#1f2933'
# 'none' by default: the inspiration/expiration label is treated as an external descriptive
# variable (cf. the comparison of encodings, Cramer's V = 0.95 in 'binary' mode)
PHASE_MODE = 'none'   # 'none', 'raw' (max_icp_phase 0-1), 'binary' (label), 'circular' (cos/sin)

cols_base = ['icp', 'icp_pulse_amplitude', 'P2P1', 'resp_in_icp']
cols_log = ['icp_pulse_amplitude', 'resp_in_icp']   # strongly skewed distributions

noms_features = {
    'icp': 'ICP',
    'icp_pulse_amplitude': 'ICP pulse amplitude (log)',
    'P2P1': 'P2/P1 ratio',
    'resp_in_icp': 'Respiratory in ICP (log)',
    'phase_cos': 'cos(peak ICP phase)',
    'phase_sin': 'sin(peak ICP phase)',
    'phase_insp': 'Peak ICP in inspiration',
    'phase_raw': 'Peak ICP phase (raw 0-1)',
}

clust = data.dropna(subset=cols_base + ['max_icp_phase', 'label_max_icp_phase', 'subject']).copy()

X = pd.DataFrame(index=clust.index)
for c in cols_base:
    X[c] = np.log1p(clust[c].clip(lower=0)) if c in cols_log else clust[c]

X = pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)

# the phase of the ICP maximum is circular (0 = 1 = start of the cycle): a cos/sin encoding
# avoids cutting the cycle artificially. Both columns are divided by sqrt(2) so that the
# phase weighs as much as a single variable in the Euclidean distance.
if PHASE_MODE == 'circular':
    X['phase_cos'] = np.cos(2 * np.pi * clust['max_icp_phase']) / np.sqrt(2)
    X['phase_sin'] = np.sin(2 * np.pi * clust['max_icp_phase']) / np.sqrt(2)
elif PHASE_MODE == 'binary':
    v = (clust['label_max_icp_phase'] == 'inspiration').astype(float)
    X['phase_insp'] = (v - v.mean()) / v.std()
elif PHASE_MODE == 'raw':
    # the bi-segment deformation already harmonises the cycles: the 0-1 phase is comparable
    # from one cycle and one patient to another, only the 0/1 junction needs care
    v = clust['max_icp_phase']
    X['phase_raw'] = (v - v.mean()) / v.std()

Xs = X.values
labels_features = [noms_features.get(c, c) for c in X.columns]

print(f'{Xs.shape[0]} windows x {Xs.shape[1]} features, {clust["subject"].nunique()} subjects')
print('features :', ', '.join(labels_features))
print(f"windows lost to missing values : {len(data) - len(clust)}")

In [ ]:
# --- how many clusters? stability criterion by patient bootstrap ---
# Silhouette and BIC judge the shape of the clusters; when the structure is weak they push
# towards the extremes (k = 2 for one, k max for the other). The relevant criterion is
# reproducibility: whole PATIENTS are resampled with replacement, k-means is rerun, and we
# check whether the partition comes back to the same thing (Hennig, clusterwise stability).
# A real cluster survives a change of sample, a chance cluster does not.
from sklearn.metrics import adjusted_rand_score

N_BOOT_K = 50            # number of draws per k: this drives the computation time
ks_stab = list(range(2, 9))

ref = {k: KMeans(n_clusters=k, n_init=20, random_state=0).fit(Xs) for k in ks_stab}
pos_sujet_clu = {s: np.flatnonzero(clust['subject'].values == s) for s in clust['subject'].unique()}
sujets_clu = np.array(list(pos_sujet_clu))
rng_stab = np.random.default_rng(0)

ari_stab = {k: [] for k in ks_stab}
jac_stab = {k: [] for k in ks_stab}

for b in range(N_BOOT_K):
    tirage = np.concatenate([pos_sujet_clu[s] for s in rng_stab.choice(sujets_clu, len(sujets_clu))])
    for k in ks_stab:
        # model fitted on the bootstrap sample, applied to ALL the original windows: the two
        # partitions then concern the same objects and are comparable
        lab_b = KMeans(n_clusters=k, n_init=3, random_state=b).fit(Xs[tirage]).predict(Xs)
        lab_ref = ref[k].labels_
        ari_stab[k].append(adjusted_rand_score(lab_ref, lab_b))
        # cluster-by-cluster stability: best Jaccard overlap found in the bootstrap
        jac_stab[k].append([max((((lab_ref == g) & (lab_b == h)).sum() /
                                 ((lab_ref == g) | (lab_b == h)).sum()) for h in range(k))
                            for g in range(k)])

ari_moy = np.array([np.mean(ari_stab[k]) for k in ks_stab])
jac_moy = {k: np.mean(jac_stab[k], axis=0) for k in ks_stab}      # one value per cluster
jac_min = np.array([jac_moy[k].min() for k in ks_stab])

fig, axs = plt.subplots(ncols=2, figsize=(14, 4.8), constrained_layout=True)

ax = axs[0]
bas = [np.percentile(ari_stab[k], 25) for k in ks_stab]
haut = [np.percentile(ari_stab[k], 75) for k in ks_stab]
ax.fill_between(ks_stab, bas, haut, color='#3b6ea5', alpha=.20)
ax.plot(ks_stab, ari_moy, 'o-', color='#3b6ea5', lw=2)
k_best = ks_stab[int(np.argmax(ari_moy))]
ax.plot(k_best, ari_moy.max(), 'o', color='#c0392b', ms=12, mfc='none', mew=2)
ax.set_title(f'A. Partition stability under patient bootstrap ({N_BOOT_K} draws)',
             color=ink, fontsize=11, loc='left')
ax.set_xlabel('Number of clusters'); ax.set_ylabel('Adjusted Rand index vs reference')
ax.set_ylim(0, 1)

ax = axs[1]
for k in ks_stab:
    ax.scatter([k] * len(jac_moy[k]), jac_moy[k], s=26, color='#9ca3af', zorder=3)
ax.plot(ks_stab, jac_min, 'o-', color='#c0392b', lw=2, label='least stable cluster')
ax.axhline(.75, color='#1f2933', lw=.8, ls='--')
ax.axhline(.60, color='#1f2933', lw=.8, ls=':')
ax.text(ks_stab[-1], .76, 'stable (0.75)  ', ha='right', va='bottom', fontsize=8, color=ink)
ax.text(ks_stab[-1], .61, 'unstable below 0.60  ', ha='right', va='bottom', fontsize=8, color=ink)
ax.set_title('B. Per-cluster stability (mean Jaccard, one dot per cluster)',
             color=ink, fontsize=11, loc='left')
ax.set_xlabel('Number of clusters'); ax.set_ylabel('Mean Jaccard recovery')
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=9, loc='lower left')

fig.suptitle('Choosing k by reproducibility rather than by cluster shape',
             color=ink, fontsize=13)
plt.show()

table_stab = pd.DataFrame({
    'mean ARI': ari_moy.round(3),
    'ARI IQR': [f'[{np.percentile(ari_stab[k], 25):.2f} - {np.percentile(ari_stab[k], 75):.2f}]'
                for k in ks_stab],
    'least stable cluster (Jaccard)': jac_min.round(3),
    'clusters with Jaccard > 0.75': [int((jac_moy[k] > .75).sum()) for k in ks_stab],
    'smallest cluster (%)': [round(pd.Series(ref[k].labels_).value_counts().min() / len(clust) * 100, 1)
                             for k in ks_stab],
}, index=pd.Index(ks_stab, name='k'))
display(table_stab)
print(f'maximal stability at k = {k_best} (mean ARI {ari_moy.max():.3f})')
print('Usual rule (Hennig): mean Jaccard > 0.75 = reproducible cluster, '
      '< 0.60 = sampling artefact. Keep the largest k for which ALL clusters '
      'stay above the threshold.')

In [ ]:
# --- how many clusters? inertia elbow, silhouette, BIC of a Gaussian mixture ---
ks = list(range(2, 9))
inerties, silhouettes, bics = [], [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(Xs, km.labels_,
                                        sample_size=min(5000, Xs.shape[0]), random_state=0))
    bics.append(GaussianMixture(n_components=k, covariance_type='full',
                                random_state=0).fit(Xs).bic(Xs))

fig, axs = plt.subplots(ncols=3, figsize=(14, 4), constrained_layout=True)
for ax, val, titre, ylab in zip(axs, [inerties, silhouettes, bics],
                                ['A. Elbow (k-means inertia)', 'B. Silhouette score', 'C. Gaussian mixture BIC'],
                                ['Within-cluster sum of squares', 'Mean silhouette', 'BIC']):
    ax.plot(ks, val, 'o-', color='#3b6ea5')
    ax.set_title(titre, color=ink, fontsize=11, loc='left')
    ax.set_xlabel('Number of clusters'); ax.set_ylabel(ylab)
axs[1].plot(ks[int(np.argmax(silhouettes))], max(silhouettes), 'o', color='#c0392b', ms=10, mfc='none')
axs[2].plot(ks[int(np.argmin(bics))], min(bics), 'o', color='#c0392b', ms=10, mfc='none')
plt.show()

print(f'best silhouette : k = {ks[int(np.argmax(silhouettes))]} '
      f'({max(silhouettes):.3f})')
print(f'minimal BIC : k = {ks[int(np.argmin(bics))]}')
print('silhouettes :', {k: round(s, 3) for k, s in zip(ks, silhouettes)})

In [ ]:
# per-cluster Jaccard stability for k = 3, with clusters ordered by increasing median ICP
med_ref = clust.assign(_l=ref[3].labels_).groupby('_l')['icp'].median().sort_values()
print(pd.Series(jac_moy[3][med_ref.index], index=[f'C{i+1}' for i in range(3)]).round(3))

In [ ]:
# --- final clustering: k-means, PCA projection and mean profile of each cluster ---
K = 3   # number of clusters retained, to adjust from the previous cells

km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(Xs)

# clusters are renumbered by increasing median ICP: the numbering becomes stable and
# interpretable from one run to the next
med_icp = clust.assign(_lab=km.labels_).groupby('_lab')['icp'].median()
remap = {ancien: f'C{i + 1}' for i, ancien in enumerate(med_icp.sort_values().index)}
clust['cluster'] = pd.Series(km.labels_, index=clust.index).map(remap)
noms_clusters = [f'C{i + 1}' for i in range(K)]
couleurs_clusters = dict(zip(noms_clusters, sns.color_palette('Set2', K)))

feat_cols = list(X.columns)
profils = (X.assign(cluster=clust['cluster']).groupby('cluster')[feat_cols]
             .mean().reindex(noms_clusters))
effectifs = clust['cluster'].value_counts().reindex(noms_clusters)

fig, axs = plt.subplots(ncols=2, figsize=(15, 5.5), width_ratios=[1.15, 1], constrained_layout=True)

# A: PCA projection (first two components), subsampled points
pca = PCA(n_components=2).fit(Xs)
proj = pca.transform(Xs)
rng = np.random.default_rng(0)
idx = rng.choice(len(proj), min(4000, len(proj)), replace=False)
ax = axs[0]
for nom in noms_clusters:
    m = (clust['cluster'].values[idx] == nom)
    ax.scatter(proj[idx][m, 0], proj[idx][m, 1], s=6, alpha=.35,
               color=couleurs_clusters[nom], edgecolor='none',
               label=f'{nom} (n = {effectifs[nom]}, {effectifs[nom] / len(clust) * 100:.0f} %)')
centres = pca.transform(km.cluster_centers_)
for ancien, nom in remap.items():
    ax.scatter(*centres[ancien], marker='X', s=180, color=couleurs_clusters[nom],
               edgecolor=ink, linewidth=1.2, zorder=5)
ax.set_title('A. PCA projection of the 60-min windows', color=ink, fontsize=11, loc='left')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.0f} % of variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.0f} % of variance)')
leg = ax.legend(frameon=False, fontsize=9, markerscale=2.5, loc='best')
for h in leg.legend_handles:
    h.set_alpha(1)

# B: standardised mean profile of each cluster (the follow-up of the heatmap)
ax = axs[1]
sns.heatmap(profils.T, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1.5, vmax=1.5,
            linewidths=.5, cbar_kws={'label': 'mean z-score'}, ax=ax)
ax.set_yticklabels(labels_features, rotation=0)
ax.set_xlabel(''); ax.set_ylabel('')
ax.set_title('B. Standardised cluster profiles', color=ink, fontsize=11, loc='left')

fig.suptitle('Unsupervised phenotyping of cerebral compliance', color=ink, fontsize=13)
plt.show()

print(f'silhouette for k = {K} : '
      f'{silhouette_score(Xs, km.labels_, sample_size=min(5000, Xs.shape[0]), random_state=0):.3f}')

In [ ]:
# --- how much does each variable weigh in the partition? PCA loadings and ablation ---
# k-means runs on the 4 standardised variables, NOT on the principal components: the PCA
# above is only the 2-D projection used for display. Standardisation gives each variable
# the same variance, not the same weight in the separation. Three complementary readings:
#   1. PCA loadings: which variables build the axes along which the clusters spread
#   2. eta2 of each variable against the clusters: share of ITS variance explained by the
#      partition, i.e. how well the clusters separate that variable; plus the permutation
#      importance of a classifier that re-learns the partition
#   3. ablation: k-means rerun without each variable. Does the partition survive (ARI vs
#      the 4-variable partition) and stay reproducible (patient bootstrap)? This is the
#      direct test of "does adding resp_in_icp improve the classification"
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import adjusted_rand_score

N_BOOT_ABL = 30          # patient-bootstrap draws per ablation configuration
lab_full = km.labels_
couleurs4 = ['#1f2933', '#3b6ea5', '#c0392b', '#5a8f5a']

# 1. PCA on the 4 standardised variables
pca4 = PCA().fit(Xs)
axes_pc = [f'PC{i + 1}' for i in range(Xs.shape[1])]
charges4 = pd.DataFrame(pca4.components_.T, index=labels_features, columns=axes_pc)
contrib4 = charges4 ** 2 * 100                              # % of each axis carried by each variable
corr_pc4 = charges4 * np.sqrt(pca4.explained_variance_)     # correlation variable / component
print('variance explained by each component (%):',
      dict(zip(axes_pc, (pca4.explained_variance_ratio_ * 100).round(1))))
print('\ncorrelation between the standardised variables:')
display(pd.DataFrame(np.corrcoef(Xs.T), index=labels_features, columns=labels_features).round(2))
print('contribution of each variable to each component (%, columns sum to 100):')
display(contrib4.round(1))

# 2. how well does the partition separate each variable?
def eta2(x, lab):
    """between-cluster sum of squares / total sum of squares of x"""
    x = np.asarray(x, float)
    between = sum(((x[lab == g].mean() - x.mean()) ** 2) * (lab == g).sum() for g in np.unique(lab))
    return between / ((x - x.mean()) ** 2).sum()

eta_full = pd.Series({nom: eta2(X[c], lab_full) for c, nom in zip(X.columns, labels_features)})
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xs, lab_full)
perm = permutation_importance(rf, Xs, lab_full, n_repeats=5, random_state=0, n_jobs=-1)
importance = pd.DataFrame({
    'eta2 (share of variance explained by clusters)': eta_full.round(3),
    'share of between-cluster inertia (%)': (eta_full / eta_full.sum() * 100).round(1),
    'permutation importance (accuracy drop)': np.round(perm.importances_mean, 3),
})
inertie_tot = ((Xs - Xs.mean(0)) ** 2).sum()
print(f'share of total inertia explained by the partition: {1 - km.inertia_ / inertie_tot:.3f}')
display(importance)

# 3. ablation
pos_sujet_abl = {s: np.flatnonzero(clust['subject'].values == s) for s in clust['subject'].unique()}
sujets_abl = np.array(list(pos_sujet_abl))

def stabilite_bootstrap(Xm, lab_ref, k, n_boot=N_BOOT_ABL, seed=0):
    """mean ARI between the reference partition and k-means refitted on patient bootstraps"""
    rng = np.random.default_rng(seed)
    aris = []
    for b in range(n_boot):
        tirage = np.concatenate([pos_sujet_abl[s] for s in rng.choice(sujets_abl, len(sujets_abl))])
        lab_b = KMeans(n_clusters=k, n_init=3, random_state=b).fit(Xm[tirage]).predict(Xm)
        aris.append(adjusted_rand_score(lab_ref, lab_b))
    return np.array(aris)

configs = {'all 4 variables': list(X.columns)}
for c in X.columns:
    configs[f'without {noms_features[c]}'] = [x for x in X.columns if x != c]

lignes_abl, partitions_abl, stab_abl = [], {}, {}
for nom, cols in configs.items():
    Xm = X[cols].values
    km_m = KMeans(n_clusters=K, n_init=20, random_state=0).fit(Xm)
    partitions_abl[nom] = km_m.labels_
    aris = stabilite_bootstrap(Xm, km_m.labels_, K)
    stab_abl[nom] = aris.mean()
    lignes_abl.append({
        'variables': len(cols),
        'ARI vs 4-variable partition': round(adjusted_rand_score(lab_full, km_m.labels_), 3),
        'silhouette (own space)': round(silhouette_score(Xm, km_m.labels_,
                                                         sample_size=min(5000, len(Xm)), random_state=0), 3),
        'bootstrap stability (mean ARI, IQR)': f'{aris.mean():.2f} [{np.percentile(aris, 25):.2f} - '
                                                f'{np.percentile(aris, 75):.2f}]',
        **{f'eta2 {noms_features[c]}': round(eta2(X[c], km_m.labels_), 2) for c in X.columns},
        'smallest cluster (%)': round(pd.Series(km_m.labels_).value_counts().min() / len(Xm) * 100, 1),
    })
ablation = pd.DataFrame(lignes_abl, index=pd.Index(configs, name='k-means on'))
print('\nablation: k-means rerun without each variable (eta2 columns: is the variable still '
      'separated by the clusters even when it is not used?)')
display(ablation)

sans_resp = f'without {noms_features["resp_in_icp"]}'
print(f'cross-table: 4-variable partition (rows) vs partition {sans_resp} (columns)')
display(pd.crosstab(clust['cluster'], pd.Series(partitions_abl[sans_resp], index=clust.index, name='no resp')))

# figure
fig, axs = plt.subplots(ncols=3, figsize=(16.5, 5), width_ratios=[1, .9, 1.2], constrained_layout=True)

# A: correlation circle, with the cluster centres in the same plane
ax = axs[0]
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='#9ca3af', lw=1))
ax.axhline(0, color='#9ca3af', lw=.8); ax.axvline(0, color='#9ca3af', lw=.8)
for j, nom in enumerate(labels_features):
    x, y = corr_pc4.loc[nom, 'PC1'], corr_pc4.loc[nom, 'PC2']
    ax.annotate('', xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=couleurs4[j], lw=2))
    # pulse amplitude and respiratory modulation point in the same direction: stagger their labels
    dy = {1: .09, 3: -.09}.get(j, 0)
    ax.text(x * 1.12, y * 1.12 + dy, nom, color=couleurs4[j], fontsize=8.5, ha='center', va='center')
centres4 = pca4.transform(km.cluster_centers_)[:, :2]
echelle = np.abs(centres4).max() / .85
for ancien, nom in remap.items():
    ax.scatter(*(centres4[ancien] / echelle), marker='X', s=130, color=couleurs_clusters[nom],
               edgecolor=ink, linewidth=1, zorder=5, label=f'{nom} centre (rescaled)')
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect('equal')
ax.set_xlabel(f'PC1 ({pca4.explained_variance_ratio_[0] * 100:.0f} %)')
ax.set_ylabel(f'PC2 ({pca4.explained_variance_ratio_[1] * 100:.0f} %)')
ax.set_title('A. Correlation circle (variable / component)', color=ink, fontsize=11, loc='left')
ax.legend(frameon=False, fontsize=8, loc='lower left')

# B: eta2 per variable
ax = axs[1]
ax.bar(range(len(eta_full)), eta_full.values, color=couleurs4, edgecolor='none', width=.7)
for j, v in enumerate(eta_full.values):
    ax.text(j, v + .02, f'{v:.2f}', ha='center', fontsize=9, color=ink)
ax.set_xticks(range(len(eta_full))); ax.set_xticklabels(labels_features, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel('eta2 (variance explained by the clusters)')
ax.set_title('B. How well the partition separates each variable', color=ink, fontsize=11, loc='left')

# C: ablation
ax = axs[2]
y = np.arange(len(ablation))
ax.barh(y - .19, ablation['ARI vs 4-variable partition'], height=.38, color='#3b6ea5',
        label='agreement with the 4-variable partition (ARI)')
ax.barh(y + .19, [stab_abl[n] for n in ablation.index], height=.38, color='#9ca3af',
        label='patient-bootstrap stability (mean ARI)')
ax.set_yticks(y); ax.set_yticklabels(ablation.index, fontsize=8); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.axvline(.75, color=ink, lw=.8, ls='--')
ax.set_title('C. Ablation: k-means without each variable', color=ink, fontsize=11, loc='left')
ax.legend(frameon=False, fontsize=8, loc='lower right')

fig.suptitle('Weight of each variable in the compliance partition', color=ink, fontsize=13)
plt.show()


In [ ]:
# --- what does each cluster look like physiologically? variables in original units ---
from scipy.stats import kruskal, chi2_contingency

variables_clusters = [
    ('icp',                 'ICP (mmHg)',                     'A'),
    ('icp_pulse_amplitude', 'ICP pulse amplitude (mmHg)',     'B'),
    ('P2P1',                'P2/P1 ratio',                    'C'),
    ('resp_in_icp',         'Respiratory in ICP (mmHg)',   'D'),
]

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(15, 9), constrained_layout=True)

for ax, (col, label, lettre) in zip(axs.flat, variables_clusters):
    sns.violinplot(data=clust, x='cluster', y=col, order=noms_clusters, hue='cluster',
                   palette=couleurs_clusters, legend=False, inner='quartile', cut=0,
                   linewidth=1, ax=ax)
    for coll in ax.collections:
        coll.set_alpha(.55)
    p = kruskal(*[clust.loc[clust['cluster'] == c, col].dropna().values for c in noms_clusters]).pvalue
    ax.set_title(f'{lettre}. {label}   {pval_stars(p)} (p{readable_pval(p)})',
                 color=ink, fontsize=11, loc='left')
    ax.set_xlabel(''); ax.set_ylabel(label)

# E: phase of the ICP maximum within the respiratory cycle (circular variable)
ax = axs.flat[4]
for nom in noms_clusters:
    v = clust.loc[clust['cluster'] == nom, 'max_icp_phase'].dropna()
    sns.kdeplot(v, ax=ax, color=couleurs_clusters[nom], lw=2, label=nom, bw_adjust=.6, clip=(0, 1))
ax.set_title('E. Phase of peak ICP within the respiratory cycle', color=ink, fontsize=11, loc='left')
ax.set_xlabel('Respiratory cycle phase (0 = start of inspiration)')
ax.set_ylabel('Density'); ax.set_xlim(0, 1)
ax.legend(frameon=False, fontsize=9)

# F: categorical composition of each cluster
ax = axs.flat[5]
comp = pd.DataFrame({
    'Peak ICP in inspiration': clust.groupby('cluster')['label_max_icp_phase']
                                    .apply(lambda s: (s == 'inspiration').mean() * 100),
    'Controlled ventilation': clust.groupby('cluster')['ventilation_mode']
                                   .apply(lambda s: (s == 'controlled').mean() * 100),
}).reindex(noms_clusters)
comp.plot.bar(ax=ax, color=['#4b5563', '#9ca3af'], width=.75, edgecolor='none', rot=0)
ax.axhline(50, color=ink, lw=.8, ls='--')
ax.set_title('F. Categorical composition of the clusters', color=ink, fontsize=11, loc='left')
ax.set_ylabel('% of windows within cluster'); ax.set_xlabel('')
ax.set_ylim(0, 108); ax.legend(frameon=False, fontsize=9, loc='upper right')

fig.suptitle('Physiological characterisation of the clusters', color=ink, fontsize=13)
plt.show()

In [ ]:
# --- descriptive table of the clusters (original units) ---
lignes_clusters = []
for col, label, _ in variables_clusters + [('time', 'Days from ICU admission', 'G')]:
    ligne = {'Variable': label}
    for nom in noms_clusters:
        v = clust.loc[clust['cluster'] == nom, col].dropna()
        ligne[nom] = f'{v.median():.1f} {iqr_interval(v, 1)}'
    p = kruskal(*[clust.loc[clust['cluster'] == c, col].dropna().values for c in noms_clusters]).pvalue
    ligne['p (Kruskal-Wallis)'] = readable_pval(p).replace(' = ', '').replace(' ', '')
    ligne['stars'] = pval_stars(p)
    lignes_clusters.append(ligne)

for col, val, label in [('label_max_icp_phase', 'inspiration', 'Peak ICP in inspiration, %'),
                        ('ventilation_mode', 'controlled', 'Controlled ventilation, %')]:
    ligne = {'Variable': label}
    for nom in noms_clusters:
        s = clust.loc[clust['cluster'] == nom, col]
        ligne[nom] = f'{(s == val).mean() * 100:.0f} %'
    p = chi2_contingency(pd.crosstab(clust['cluster'], clust[col]).values)[1]
    ligne['p (Kruskal-Wallis)'] = readable_pval(p).replace(' = ', '').replace(' ', '')
    ligne['stars'] = pval_stars(p)
    lignes_clusters.append(ligne)

ligne = {'Variable': 'Windows, n (%)'}
for nom in noms_clusters:
    ligne[nom] = f'{effectifs[nom]} ({effectifs[nom] / len(clust) * 100:.0f} %)'
ligne['p (Kruskal-Wallis)'] = ''; ligne['stars'] = ''
lignes_clusters.insert(0, ligne)

ligne = {'Variable': 'Subjects contributing, n'}
for nom in noms_clusters:
    ligne[nom] = f"{clust.loc[clust['cluster'] == nom, 'subject'].nunique()}"
ligne['p (Kruskal-Wallis)'] = ''; ligne['stars'] = ''
lignes_clusters.insert(1, ligne)

table_clusters = pd.DataFrame(lignes_clusters).set_index('Variable')
print('Median [IQR] unless stated otherwise. Chi-square test for categorical rows.')
print('Significance: * p < 0.05, ** p < 0.01, *** p < 0.001, ns = not significant.\n')
display(table_clusters)

In [ ]:
# --- are the clusters physiological profiles or simply patients? ---
# a window is not independent of the other windows of the same patient: if each patient
# lives in a single cluster, we have clustered patients, not compliance states.
compo = (pd.crosstab(clust['subject'], clust['cluster'], normalize='index')
           .reindex(columns=noms_clusters).fillna(0) * 100)
part_dominante = compo.max(axis=1)
hasard = effectifs.max() / len(clust) * 100    # expected share if the windows were shuffled

ordre_sujets = compo.assign(dom=compo.idxmax(axis=1), part=part_dominante) \
                    .sort_values(['dom', 'part'], ascending=[True, False]).index

fig, axs = plt.subplots(nrows=2, figsize=(14, 8), height_ratios=[1, 1.1], constrained_layout=True)

ax = axs[0]
bas = np.zeros(len(ordre_sujets))
for nom in noms_clusters:
    v = compo.loc[ordre_sujets, nom].values
    ax.bar(np.arange(len(ordre_sujets)), v, bottom=bas, width=1, color=couleurs_clusters[nom], label=nom)
    bas += v
ax.set_xlim(-.5, len(ordre_sujets) - .5); ax.set_ylim(0, 100)
ax.set_xticks([])
ax.set_title('A. Cluster composition of each patient (one bar = one patient)',
             color=ink, fontsize=11, loc='left')
ax.set_xlabel(f'Patients (n = {len(ordre_sujets)}), sorted by dominant cluster')
ax.set_ylabel('% of the patient windows')
ax.legend(frameon=False, fontsize=9, ncol=len(noms_clusters), loc='lower center',
          bbox_to_anchor=(.5, 1.02))

ax = axs[1]
ax.hist(part_dominante, bins=np.arange(0, 105, 5), color='#3b6ea5', edgecolor='white')
ax.axvline(hasard, color='#c0392b', lw=2, ls='--')
ax.set_ylim(0, ax.get_ylim()[1] * 1.18)
ax.text(hasard, ax.get_ylim()[1] * .97, f'chance level ({hasard:.0f} %)  ',
        color='#c0392b', fontsize=9, ha='right', va='top')
ax.set_xlim(0, 100)
ax.set_title('B. Share of a patient windows falling in their dominant cluster',
             color=ink, fontsize=11, loc='left')
ax.set_xlabel('% of windows in the dominant cluster'); ax.set_ylabel('Number of patients')

fig.suptitle('How much of the clustering is driven by between-patient differences?',
             color=ink, fontsize=13)
plt.show()

n_mono = int((part_dominante > 80).sum())
print(f'median dominant share : {part_dominante.median():.0f} % {iqr_interval(part_dominante, 0)} '
      f'(chance level : {hasard:.0f} %)')
print(f'{n_mono}/{len(part_dominante)} patients spend more than 80 % of their monitoring '
      f'in a single cluster')
print("the further the median is from chance, the more the clusters separate patients "
      "rather than successive physiological states of a given patient")

In [ ]:
# --- patient view: hierarchical clustering of the median profile of each patient ---
# the natural follow-up of the heatmap: one row per patient, their compliance variables
# summarised by their median, grouped by agglomerative hierarchical clustering
profil_sujet = (X.assign(subject=clust['subject']).groupby('subject')[feat_cols].median())

cm = sns.clustermap(profil_sujet, method='ward', metric='euclidean',
                    cmap='vlag', center=0, vmin=-1.5, vmax=1.5,
                    figsize=(8, 11), yticklabels=True,
                    cbar_kws={'label': 'median z-score'},
                    dendrogram_ratio=(.18, .10))
cm.ax_heatmap.set_xticklabels(labels_features, rotation=35, ha='right')
cm.ax_heatmap.set_ylabel(f'Patients (n = {len(profil_sujet)})')
cm.ax_heatmap.set_xlabel('')
cm.ax_heatmap.tick_params(labelsize=7)
cm.ax_col_dendrogram.set_title('Patient-level compliance profiles (hierarchical clustering)',
                               color=ink, fontsize=13, pad=14)
plt.show()

# patient groups cut from the dendrogram, to compare with the window clusters
from scipy.cluster.hierarchy import fcluster
K_SUJETS = 3   # number of patient groups cut from the dendrogram
groupes_sujets = pd.Series(fcluster(cm.dendrogram_row.linkage, K_SUJETS, criterion='maxclust'),
                           index=profil_sujet.index, name='patient_group')
clust['patient_group'] = clust['subject'].map(groupes_sujets)

print(f'{len(profil_sujet)} patients split into {K_SUJETS} groups : '
      f'{groupes_sujets.value_counts().sort_index().to_dict()}')
display(pd.crosstab(clust['patient_group'], clust['cluster'], normalize='index')
          .reindex(columns=noms_clusters).mul(100).round(0)
          .rename_axis(index='patient group', columns='window cluster (%)'))

In [ ]:
# --- trajectories: patients who change cluster during the monitoring ---
# "mixed" patients are those who spend less than SEUIL_MIXTE % of their windows in a single
# cluster: they are the only ones in whom the cluster is a state and not a trait
SEUIL_MIXTE = 90

sujets_mixtes = part_dominante[part_dominante < SEUIL_MIXTE].sort_values().index
traj = clust[clust['subject'].isin(sujets_mixtes)].sort_values(['subject', 'time'])

# display order: by cluster of the first day, then by increasing dominant share
premier = traj.groupby('subject').first()['cluster']
ordre_traj = (pd.DataFrame({'premier': premier, 'part': part_dominante[sujets_mixtes]})
                .sort_values(['premier', 'part']).index)
rang = {s: i for i, s in enumerate(ordre_traj)}

# height of panel A proportional to the number of mixed patients, otherwise the rows
# overlap as soon as the threshold keeps many of them
h_a = max(3.5, .20 * len(ordre_traj))
fig, axs = plt.subplots(nrows=2, figsize=(14, h_a + 3.4), height_ratios=[h_a, 3.4],
                        constrained_layout=True)

# A: one row per patient, one tile per 60-min window, coloured by cluster
ax = axs[0]
for nom in noms_clusters:
    m = traj['cluster'] == nom
    ax.bar(traj.loc[m, 'time'], height=.82, bottom=traj.loc[m, 'subject'].map(rang) - .41,
           width=1 / 24, color=couleurs_clusters[nom], label=nom, align='edge')
ax.set_ylim(-.6, len(ordre_traj) - .4)
ax.set_yticks(range(len(ordre_traj)))
ax.set_yticklabels([f'{s} ({part_dominante[s]:.0f} %)' for s in ordre_traj], fontsize=7)
ax.set_xlabel('Days from ICU admission')
ax.set_ylabel('Patients (dominant cluster share)')
ax.set_title(f'A. Cluster trajectory of the {len(ordre_traj)} mixed patients '
             f'(< {SEUIL_MIXTE} % of windows in a single cluster), one tile = one 60-min window',
             color=ink, fontsize=11, loc='left')
ax.legend(frameon=False, fontsize=9, ncol=len(noms_clusters), loc='lower center',
          bbox_to_anchor=(.5, 1.03))

# B: prevalence of each cluster over the stay, whole cohort
ax = axs[1]
bornes = np.arange(0, np.ceil(clust['time'].max()) + 1, 1)
jour = pd.cut(clust['time'], bornes, labels=bornes[:-1], right=False)
prev = (pd.crosstab(jour, clust['cluster'], normalize='index')
          .reindex(columns=noms_clusters).fillna(0) * 100)
n_par_jour = jour.value_counts().sort_index()
garde = n_par_jour[n_par_jour >= 50].index          # days with too little coverage: not drawn
prev = prev.loc[prev.index.isin(garde)]
ax.stackplot(prev.index.astype(float), *[prev[c].values for c in noms_clusters],
             colors=[couleurs_clusters[c] for c in noms_clusters], labels=noms_clusters, alpha=.9)
ax.set_xlim(prev.index.astype(float).min(), prev.index.astype(float).max())
ax.set_ylim(0, 100)
ax.set_xlabel('Days from ICU admission'); ax.set_ylabel('% of windows')
ax.set_title('B. Cluster prevalence over the ICU stay, whole cohort '
             '(days with at least 50 windows)', color=ink, fontsize=11, loc='left')

fig.suptitle('Do compliance profiles change over time?', color=ink, fontsize=13)
plt.show()

# temporal stability: length of the runs spent in a same cluster.
# Counted in consecutive windows of the table: discarded windows (missing data) create
# gaps, so a run is not necessarily continuous in time.
runs = []
for sub_id, g in clust.sort_values(['subject', 'time']).groupby('subject'):
    c = g['cluster'].values
    changements = np.flatnonzero(c[1:] != c[:-1])
    longueurs = np.diff(np.concatenate(([-1], changements, [len(c) - 1])))
    runs.append({'subject': sub_id, 'n_fenetres': len(c),
                 'n_transitions': len(changements),
                 'run_median': float(np.median(longueurs)),
                 'mixte': sub_id in set(sujets_mixtes)})
runs = pd.DataFrame(runs)

print(f'{len(sujets_mixtes)}/{len(part_dominante)} mixed patients (< {SEUIL_MIXTE} % in one cluster)')
for est_mixte, g in runs.groupby('mixte'):
    etiquette = 'mixed ' if est_mixte else 'stable'
    print(f'{etiquette} (n = {len(g)}) : median stay in a same cluster = '
          f'{g["run_median"].median():.1f} consecutive windows (~{g["run_median"].median():.0f} h), '
          f'{g["n_transitions"].median():.0f} transitions per patient (median)')